# A field computed by DFT

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/igbaskin/cmf-qsar/blob/main/docs/tutorials/23_a_field_from_dft.ipynb)

Every field so far was built from the atoms: a Gaussian on each atom, weighted by a charge, a
size or a hydrophobicity. A quantum-chemical calculation gives fields of its own, directly as
values on a grid of points: the **electron density**, the **electron localisation function**
(ELF, a number between 0 and 1 that is large where electron pairs are localised, as in bonds and
lone pairs) and the **electrostatic potential**. `cmfqsar/gridfields.py` reads such grids and
compares two structures by the overlap of their grids, as the program compares two sets of
atom-centred fields.

The data are the 1,462 ABO<sub>3</sub> perovskite cells of the design-loop notebook, with their
formation energy. Each cell was given one static DFT calculation (density functional theory) in
VASP: PBE, the same settings for every cell, no relaxation. That took 2 hours 15 minutes on this
machine and left 2 GB of grids, which are not in the repository. So this notebook reads the
results of `benchmarks/run_gridfields.py` for the grids, and computes live only what needs no
grid.

The notebook asks three questions.

1. Does the grid know more than its own fit with Gaussians on the atoms? A fitted grid has a
   closed-form kernel, like every field of the program, so this decides whether the grid is
   worth its cost as a *field*.
2. Does it predict the formation energy better than the program's own fields, or the elements
   alone?
3. The reference: the DFT run that gives the grid of a cell also gives its **total energy**, and
   a formation energy is a total energy minus the energies of the elements. Does the grid beat a
   linear regression on that one number and the element counts?

Running time: about one minute. The first run downloads 4 MB of perovskites, and the grids are
not needed.

In [1]:
# Run this cell first. On Google Colab it fetches the repository and installs the package;
# on a local checkout it only moves to the repository root, so that the example paths work.
import os, subprocess, sys

if "google.colab" in sys.modules:
    if not os.path.exists("cmf-qsar"):
        subprocess.run(["git", "clone", "-q", "https://github.com/igbaskin/cmf-qsar.git"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "./cmf-qsar", "py3Dmol"], check=True)
    os.chdir("cmf-qsar")
else:
    while not os.path.exists("pyproject.toml") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")
print("working in", os.getcwd())

working in C:\Users\baski\work\claude\cmf


## The grids and their fits

Each cell gives three grids, put on 24 × 24 × 24 points of the cell. Two kernels compare them.

- **grid overlap**: all cells are placed on one lattice (they are all cubic), and two cells are
  compared by the overlap integral of their grids, point by point. This is the overlap kernel of
  the program, computed on the grid instead of in closed form.
- **fitted grid overlap**: each grid is first replaced by its best fit with Gaussians of three
  widths on every atom. What the fit cannot express, for example density between two atoms,
  is lost. A second fit with seven widths loses less.

The references are the four atom-centred fields of item M4 on the same lattice (electrostatic,
steric, electronegativity and the inverse square root of the mass), and the composition alone:
a kernel that knows only which elements a cell holds. Five folds repeated three times; the
ridge, and for the fields the attenuation, are chosen inside each training part.

In [2]:
import json
import numpy as np

with open("benchmarks/results/gridfields/crystals.json", encoding="utf-8") as fh:
    crystals = json.load(fh)
models = crystals["models"]
print("%d cells; formation energy %+.2f eV on the average, spread %.2f eV"
      % (crystals["cells"], crystals["y_mean"], crystals["y_sd"]))
print()
print("%-44s %8s %10s" % ("kernel, all on one lattice", "R2", "RMSE, eV"))
for name in ("grid overlap", "grid fitted with seven widths overlap", "fitted grid overlap",
             "grid elf overlap", "grid potential overlap", "grid density overlap",
             "fields overlap", "composition"):
    m = models[name]
    print("%-44s %+8.3f %10.3f" % (name, m["q2_mean"], m["rmse_mean"]))
print()
print("relative error of the fit with three widths:",
      ", ".join("%s %.0f %%" % (k, 100 * v) for k, v in crystals["fit_error_mean"].items()))
print("relative error of the fit with seven widths:",
      ", ".join("%s %.0f %%" % (k, 100 * v) for k, v in crystals["wide_error_mean"].items()))
print()
for c in crystals["comparisons"]:
    if "overlap" in c["model"] and "radial" not in c["model"]:
        print("%-58s %+.3f [%+.3f, %+.3f]" % ("%s minus %s" % (c["model"], c["against"]), c["mean"], c["lo"], c["hi"]))

1462 cells; formation energy +1.43 eV on the average, spread 0.71 eV

kernel, all on one lattice                         R2   RMSE, eV
grid overlap                                   +0.825      0.298
grid fitted with seven widths overlap          +0.754      0.353
fitted grid overlap                            +0.615      0.442
grid elf overlap                               +0.646      0.424
grid potential overlap                         +0.644      0.425
grid density overlap                           +0.513      0.497
fields overlap                                 +0.424      0.541
composition                                    +0.362      0.569

relative error of the fit with three widths: density 9 %, elf 21 %, potential 20 %
relative error of the fit with seven widths: density 3 %, elf 13 %, potential 7 %

grid overlap minus fields overlap                          +0.409 [+0.365, +0.457]
grid overlap minus fitted grid overlap                     +0.216 [+0.182, +0.248]
fields and g

**As a field, the grid is the best description of the cells.** Its overlap kernel reaches
R² 0.825, against 0.424 for the four atom-centred fields and 0.362 for the elements alone.

**It knows more than its fit on the atoms.** It beats its fit with three widths by 0.216, and the
interval lies above zero. That fit leaves a fifth of the ELF and of the potential unexplained, so
the check was repeated with seven widths. The grid still gains 0.069, with the interval above
zero. About two thirds of the first margin came from the weakness of the fit. The rest is in the
part of the grid that Gaussians on the atoms cannot reproduce, such as density between the atoms.

Each grid alone is weaker than the three together: the ELF and the potential reach about 0.65,
the density 0.51. Without a shared lattice, as radial correlations, the grid loses to its fit
and to the fields; that table is in `benchmarks/results/gridfields/table.md`.

So the question of item M9 has a positive answer on one lattice. The grid is a field worth more
than its closed-form fit. Whether it is worth its cost as a *predictor* is a different question.

## The reference: the energy of the same run

The grid of a cell is not free. It costs one DFT calculation of that cell, and the calculation
gives more than the grid: it also gives the **total energy** of the cell. The target is a
formation energy, which is a total energy minus the energies of the elements in their standard
states. So the obvious reference is a linear regression of the target on the total energy of the
run and the counts of each element. It needs nothing that the grid does not already need.

The total energies were read from the output of each run (`run_gridfields.py --stage energies`)
and are in the repository. The cell runs the regression live, as a ridge regression on the
standardised columns, with the same folds and the same choice of the ridge as the grid models.
It also runs the energy alone. The composition kernel is in the table above.

In [3]:
import time
sys.path.insert(0, "benchmarks")
import run_gridfields as GF                       # the benchmark script of the grids

start = time.time()
atoms, names, y, _prepared = GF.m4_set()           # the perovskite cells; downloads 4 MB on the first run
with open("benchmarks/results/gridfields/reference.json", encoding="utf-8") as fh:
    stored = json.load(fh)
energy_names = {}
with open(GF.ENERGIES, encoding="utf-8") as fh:
    next(fh)
    for line in fh:
        energy_names[line.split(",")[0]] = True
keep = [i for i, n in enumerate(names) if GF.safe(n) in energy_names]     # the cells with a converged run
cells = [atoms[i] for i in keep]
cell_names = [names[i] for i in keep]
y = y[keep]
E, C, elements = GF.run_energy_columns(cell_names, cells)
print("%d cells with a converged run, %d elements" % (len(y), len(elements)))

live = GF.cross_validate({
    "energy of the run": ("sum", GF.linear_kernel(E[:, None])),
    "energy of the run and element counts": ("sum", GF.linear_kernel(np.c_[E, C])),
}, y, {})
print()
print("%-40s %8s %10s %12s" % ("model", "R2", "RMSE, eV", "stored R2"))
for name, r in live.items():
    ref = stored["models"].get(name, models.get(name))
    print("%-40s %+8.3f %10.3f %+12.3f" % (name, r["q2_mean"], r["rmse_mean"], ref["q2_mean"]))
print("%-40s %+8.3f %10.3f   (stored; needs the grids)" % ("grid overlap", models["grid overlap"]["q2_mean"],
                                                         models["grid overlap"]["rmse_mean"]))
print()
for c in stored["comparisons"]:
    print("%s minus %s: %+.3f [%+.3f, %+.3f]" % (c["model"], c["against"], c["mean"], c["lo"], c["hi"]))
print("slope of the target on the energy, with the element counts: %.3f" % stored["slope_on_energy"])
print("(%.0f s)" % (time.time() - start))

1462 cells with a converged run, 56 elements



model                                          R2   RMSE, eV    stored R2
energy of the run                          +0.380      0.561       +0.380
energy of the run and element counts       +0.940      0.175       +0.940
grid overlap                               +0.825      0.298   (stored; needs the grids)

energy of the run and element counts minus grid overlap: +0.103 [+0.084, +0.127]
grid overlap and energy of the run minus energy of the run and element counts: +0.014 [+0.011, +0.017]
slope of the target on the energy, with the element counts: 0.172
(49 s)


The live numbers repeat the stored ones exactly.

**The energy of the run beats the grid.** With the element counts it reaches R² 0.940 and an
RMSE of 0.175 eV, against 0.825 and 0.298 eV for the grid. The difference is 0.103
[0.084, 0.127]. Neither part works alone: the energy alone reaches 0.380 and the elements alone
0.362. The formation energy is the energy minus the elements, and the regression needs both.

**The grid adds a little to the reference.** The two kernels summed gain 0.014 [0.011, 0.017] in
R² over the reference alone. That gain is real, and it is small.

The slope of the target on the energy is 0.17, not the 1 that the definition of a formation
energy suggests. The run is not the calculation behind the target: the target comes from another
code and functional (Castelli et al. 2012), and this run is a static PBE calculation without spin.
The regression learns how the two calculations relate, and about 1,170 training cells are enough
for that.

So the grid is a better *field* than any atom-centred one, and a worse *predictor* of an energy
than the one number its own calculation already prints. A grid model of a DFT energy is not
worth its cost here. A grid could still be worth its cost for a target that the calculation does
not give, such as a measured property. That was not tested.

## Molecules

The same comparison was made on four series of corrosion inhibitors, with the density and the
electrostatic potential of each molecule from an ORCA calculation. The series have 20 to 29
molecules. The cell reads the stored results.

In [4]:
print("%-20s %4s %14s %20s %16s" % ("series", "n", "grid overlap", "fitted grid overlap", "fields overlap"))
for s in GF.SETS:
    with open("benchmarks/results/gridfields/molecules_%s.json" % s, encoding="utf-8") as fh:
        r = json.load(fh)
    m = r["models"]
    print("%-20s %4d %+14.3f %+20.3f %+16.3f" % (s, r["molecules"], m["grid overlap"]["q2_mean"],
                                                m["fitted grid overlap"]["q2_mean"], m["fields overlap"]["q2_mean"]))

series                  n   grid overlap  fitted grid overlap   fields overlap
7_c1nncs1              29         -0.159               -0.170           -0.275
12_c1ccoc1             27         -0.097               +0.058           -0.133
13_c1c_nH_nn1          25         +0.600               +0.607           +0.602
161_c1ccc2scnc2c1      20         +0.562               +0.635           +0.580


On molecules the grid gains nothing. On no series is the grid better than its fit, and on
`12_c1ccoc1` it is worse. Against the five atom-centred fields it is level on every series. On
`7_c1nncs1` no kernel predicts better than the mean, so its numbers say nothing. With 20 to 29
molecules most intervals are wider than 0.1 (see the table of the benchmark), so small
differences cannot be seen. This repeats the conceptual-DFT study, which found that an exact
grid kernel predicted no better than the same field fitted on the atoms.

## What to take from this notebook

- A grid from DFT can be compared like any field: by the overlap of two grids on one lattice.
  On these perovskites that kernel is the best field description, better than its own fit with
  Gaussians on the atoms and better than the program's atom-centred fields.
- Before a grid is used as a predictor, ask what else the calculation gives. Here the total
  energy of the same run, with the element counts, predicts the formation energy better
  (R² 0.940 against 0.825), and the grid adds 0.014 to it.
- Without a shared lattice, and on molecules, the grid gains nothing over its fit.
- The rule: when an input costs a calculation, the reference is a regression on the numbers the
  calculation already gives, not only a model without the calculation.

The grids need VASP and about two hours of calculation; the stages are in the docstring of the
benchmark:

```
python benchmarks/run_gridfields.py --stage energies --vasp-dir DIR    # the energy of each run
python benchmarks/run_gridfields.py --stage crystals --stage reference --stage molecules
```

Where to read further in the manual: *Fields from DFT on a grid* has the module, the readers of
cube and VASP files, both kernels and the full tables. *A field is a kernel mean embedding*
explains why the overlap of two grids is the same kind of kernel as the overlap of two sets of
Gaussians.